# LLMSlim round-3 benchmark builds: `llmslim_v2`

This notebook builds the LLMSlim arms (lexical and reranker) for LoCoMo (1,540 questions) and LongMemEval-S (180) with LLMSlim as changed in round 3 (turn-by-turn transcripts, PR #48), frozen before any test result was read. NaturalQuestions and HotpotQA output is unchanged by that PR.

It **never calls a model API**. It only produces the exact context each system would send to the reader, so anyone can rerun it and compare the outputs byte for byte. The reader (Claude Haiku 5.5) and the judge run on these published contexts afterwards.

- **Code:** [Thanatos9404/llmslim](https://github.com/Thanatos9404/llmslim) at commit `6b03a339b4c0a4dcca1b16b4dae4be2f08cc4a9c`, `benchmarks/live/round3/`. Splits were committed before any result: `splits_r3.json`.
- **Data:** downloaded from each dataset's original source (lost-in-the-middle, HotpotQA, LoCoMo, LongMemEval) and checked by sha256.
- **Environments:** pinned Python 3.12 venvs (`kaggle_run.py setup`), separate from Kaggle's image.
- **Output:** `round3_llmslim_v2.zip`: one JSONL per system and suite (`id`, `context`, `tokens`, `build_ms`) and `manifest-llmslim_v2.json` (commit, GPUs, package versions, data and output hashes).

Settings: accelerator **None** (CPU is enough), internet **on**. Runtime: about 1 hour. A rerun resumes from the cases already built.

In [ ]:
import subprocess
import sys

COMMIT = "6b03a339b4c0a4dcca1b16b4dae4be2f08cc4a9c"
JOB = "llmslim_v2"
R = "/tmp/src/benchmarks/live/round3/kaggle_run.py"
subprocess.run(
    "rm -rf /tmp/src && git init -q /tmp/src && cd /tmp/src && git remote add origin https://github.com/Thanatos9404/llmslim && git fetch -q --depth 1 origin "
    + COMMIT
    + " && git checkout -q FETCH_HEAD && git log -1 --format='%H %s'",
    shell=True,
    check=True,
)
subprocess.run([sys.executable, R, "setup", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "data", "--work", "/tmp/r3"], check=True)
subprocess.run([sys.executable, R, "build", "--work", "/tmp/r3", "--job", JOB], check=True)
subprocess.run(
    [
        sys.executable,
        R,
        "package",
        "--work",
        "/tmp/r3",
        "--job",
        JOB,
        "--zip",
        "/kaggle/working/round3_" + JOB + ".zip",
    ],
    check=True,
)